# import libs

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from xgboost import XGBClassifier
import numpy as np
import joblib
import json
import os
import shap
import scorecardpy as sc
import itertools
from joblib import Parallel, delayed
from optbinning import BinningProcess
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

In [2]:
X_train = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/X_train.csv")
X_test  = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/X_test.csv")
X_oot   = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/X_oot.csv")

y_train = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/y_train.csv").squeeze()
y_test  = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/y_test.csv").squeeze()
y_oot   = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/y_oot.csv").squeeze()

In [3]:
encoders   = joblib.load('R:/some-research-about-approve-model-in-banking/models/cat_encoders.pkl')
process    = joblib.load('R:/some-research-about-approve-model-in-banking/models/binning_process.pkl')

with open('R:/some-research-about-approve-model-in-banking/models/xgb_selected_features.json') as f:
    selected = json.load(f)

In [4]:
cat = X_train.select_dtypes(include=['object']).columns.tolist()

encoders = {}
for col in cat:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    encoders[col] = le

joblib.dump(encoders, 'R:/some-research-about-approve-model-in-banking/models/cat_encoders.pkl')
print(f"Encoded {len(cat)} categorical columns: {cat}")

Encoded 16 categorical columns: ['name_contract_type', 'code_gender', 'flag_own_car', 'flag_own_realty', 'name_type_suite', 'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'occupation_type', 'weekday_appr_process_start', 'organization_type', 'fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']


In [5]:
for col in cat:
    X_test[col] = encoders[col].transform(X_test[col].astype(str))
    X_oot[col]  = encoders[col].transform(X_oot[col].astype(str))

In [6]:
train_with_label = X_train.copy()
train_with_label['target'] = y_train.values

test_with_label = X_test.copy()
test_with_label['target'] = y_test.values

oot_with_label = X_oot.copy()
oot_with_label['target'] = y_oot.values

In [7]:
for segment in train_with_label['flag_own_car'].unique():
    seg_data = train_with_label[train_with_label['flag_own_car'] == segment]
    print(f"{segment}: n={len(seg_data)}, default_rate={seg_data['target'].mean():.3f}")

0: n=138105, default_rate=0.085
1: n=71002, default_rate=0.073


# training with full dataset and try with segmentation by name_housing_type, name_education_type

In [8]:
def calc_gini(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    return 2 * auc - 1

In [9]:
full_model = XGBClassifier()

full_model.fit(X_train[selected],y_train)

gini_train = calc_gini(y_train, full_model.predict_proba(X_train[selected])[:,1])
gini_test  = calc_gini(y_test,  full_model.predict_proba(X_test[selected])[:,1])
gini_oot   = calc_gini(y_oot,   full_model.predict_proba(X_oot[selected])[:,1])

print(f"full dataset:")
print(f"gini_train: {round(gini_train, 5)}")
print(f"gini_test: {round(gini_test, 5)}")
print(f"gini_oot: {round(gini_oot, 5)}")
print(f"metric drop: {round(1- gini_oot/gini_train,5)} from gini_train = {round(gini_train, 5)}")
print(f"{'-'*50}")

for i in X_train['flag_own_car'].unique():
    sub_model = XGBClassifier()
    
    train = train_with_label[train_with_label['flag_own_car'] == i]
    test  = test_with_label[test_with_label['flag_own_car'] == i]
    oot   = oot_with_label[oot_with_label['flag_own_car'] == i]

    x_train = train.drop('target', axis=1)
    y_train_ = train['target']
    x_test  = test.drop('target', axis=1)
    y_test_  = test['target']
    x_oot   = oot.drop('target', axis=1)
    y_oot_   = oot['target']

    sub_model.fit(x_train[selected], y_train_)

    gini_train = calc_gini(y_train_, sub_model.predict_proba(x_train[selected])[:,1])
    gini_test  = calc_gini(y_test_,  sub_model.predict_proba(x_test[selected])[:,1])
    gini_oot   = calc_gini(y_oot_,   sub_model.predict_proba(x_oot[selected])[:,1])

    print(f"segment name_housing_type = {i}:")
    print(f"gini_train: {round(gini_train, 5)}")
    print(f"gini_test: {round(gini_test,5)}")
    print(f"gini_oot: {round(gini_oot, 5)}")
    print(f"metric drop: {round(1- gini_oot/gini_train,5)} from gini_train = {round(gini_train, 5)}")
    print(f"{'-'*50}")
    

full dataset:
gini_train: 0.75889
gini_test: 0.51486
gini_oot: 0.51331
metric drop: 0.3236 from gini_train = 0.75889
--------------------------------------------------
segment name_housing_type = 0:
gini_train: 0.80778
gini_test: 0.5065
gini_oot: 0.50875
metric drop: 0.37019 from gini_train = 0.80778
--------------------------------------------------
segment name_housing_type = 1:
gini_train: 0.93046
gini_test: 0.46723
gini_oot: 0.44714
metric drop: 0.51944 from gini_train = 0.93046
--------------------------------------------------


# explain about smaller sample → less variance → inflated AUC

when dividing a large dataset into smaller segments, the reduced sample size makes the model highly susceptible to noise

- This leads to a falsely high training metric that drops dramatically on the OOT set
- Ultimately, over-segmentation causes inadvertent overfitting—a classic trap of metric chasing